# Mini project by Python documentation
## Mini-Project 1: Advanced Python Data Exploration & Automated Reporting

### 1. Project Overview
This project analyzes the Sample Superstore 2019 dataset using Python.
The main goal is to explore sales, profit, customers, discounts, returns, and business performance, then generate useful analytical insights and visualizations.

The project was implemented using:
* Python
* Pandas
* NumPy
* Matplotlib
* Seaborn

### 2. Dataset
The Excel file contains three sheets:
* **Orders**: Contains the main transaction data (9,994 records, 21 columns).
* **People**: Contains regional sales managers (4 records).
* **Returns**: Contains returned orders (800 records).

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

# Load the three sheets from the Excel file
file_path = 'Sample - Superstore 2019.xls'
orders = pd.read_excel(file_path, sheet_name='Orders')
people = pd.read_excel(file_path, sheet_name='People')
returns = pd.read_excel(file_path, sheet_name='Returns')

print(f"Orders Shape: {orders.shape}")
print(f"People Shape: {people.shape}")
print(f"Returns Shape: {returns.shape}")

### 3. Data Inspection & 4. Data Cleaning
The dataset was inspected for missing values and duplicates.
* **Date Conversion**: Converted `Order Date` and `Ship Date` to datetime.
* **Missing Values**: Handled missing values in `Postal Code` using `fillna(0)`.
* **Duplicate Records**: Checked and dropped duplicate records.

In [ ]:
# Data Inspection
print("Orders info:")
orders.info()
print("\nMissing values:\n", orders.isnull().sum())
print("\nDuplicate records:", orders.duplicated().sum())

# Data Cleaning
orders = orders.drop_duplicates()
orders["Order Date"] = pd.to_datetime(orders["Order Date"])
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"])
orders["Postal Code"] = orders["Postal Code"].fillna(0)

print("\nCleaning Completed. Current Shape:", orders.shape)

### 5. Feature Engineering
New columns were created to make the analysis more useful:
* **Profit Margin**: `(Profit / Sales) * 100`
* **Shipping Duration**: `Ship Date - Order Date` (in days)
* **Sales Performance**: Classified into `Low`, `Medium`, `High`, `Very High` using `pd.cut()`
* **Year**: Extracted year from `Order Date`

In [ ]:
orders["Profit Margin"] = (orders["Profit"] / orders["Sales"]) * 100
orders["Shipping Duration"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Sales Performance"] = pd.cut(
    orders["Sales"],
    bins=[-np.inf, 50, 200, 500, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)
orders["Year"] = orders["Order Date"].dt.year

orders[['Sales', 'Profit', 'Profit Margin', 'Shipping Duration', 'Sales Performance', 'Year']].head()

### 6. KPI Analysis
Several important business KPIs were calculated to give a quick overview of performance.

In [ ]:
total_sales = orders["Sales"].sum()
total_profit = orders["Profit"].sum()
total_quantity = orders["Quantity"].sum()
total_orders = orders["Order ID"].nunique()
profit_margin = (total_profit / total_sales) * 100
avg_order_value = total_sales / total_orders

kpi_summary = pd.DataFrame({
    'KPI': ['Total Sales', 'Total Profit', 'Total Quantity', 'Total Orders', 'Profit Margin (%)', 'Average Order Value ($)'],
    'Value': [f"${total_sales/1e6:.2f}M", f"${total_profit/1e3:.2f}K", f"{total_quantity:,}", f"{total_orders:,}", f"{profit_margin:.2f}%", f"${avg_order_value:.2f}"]
})

kpi_summary

### 7. Category, 8. Region, 9. Customer Segment & 10. Discount Analysis
* **Technology** generated highest sales and profit; Furniture had a much lower profit margin.
* **West** region generated highest sales and profit; Central had the lowest profit margin.
* **Consumer** generates highest sales, while **Home Office** achieved the highest profit margin.
* **Discount Analysis**: Strong relationship between high discounts (>20%) and negative profitability.

In [ ]:
print("--- 7. Category Analysis ---")
print(orders.groupby('Category')[['Sales', 'Profit']].sum())

print("\n--- 8. Region Analysis ---")
print(orders.groupby('Region')[['Sales', 'Profit']].sum())

print("\n--- 9. Customer Segment Analysis ---")
print(orders.groupby('Segment')[['Sales', 'Profit']].sum())

print("\n--- 10. Discount Analysis ---")
discount_bins = [0, 0.1, 0.2, 0.3, 0.5, 1.0]
discount_labels = ['0–10%', '10–20%', '20–30%', '30–50%', '50%+']
orders['Discount Group'] = pd.cut(orders['Discount'], bins=discount_bins, labels=discount_labels, include_lowest=True)

discount_summary = orders.groupby('Discount Group', observed=True).apply(
    lambda x: pd.Series({
        'Sales': x['Sales'].sum(),
        'Profit': x['Profit'].sum(),
        'Profit Margin (%)': (x['Profit'].sum() / x['Sales'].sum()) * 100
    })
)
print(discount_summary)

### 11. Time, 12. Correlation, 13. Returns & 14. Outlier Analysis
* **Time Analysis**: Sales and profit increased considerably from 2016 to 2019.
* **Correlation**: Identifying relationships between numerical variables.
* **Returns**: Connected Returns sheet with Orders dataset (800 returned orders, ~8% return rate).
* **Outliers**: Identified unusual sales transactions using the IQR method.

In [ ]:
# 11. Year Analysis
print("--- 11. Sales & Profit by Year ---")
print(orders.groupby('Year')[['Sales', 'Profit']].sum())

# 12. Correlation Analysis
print("\n--- 12. Correlation Matrix ---")
num_cols = ["Sales", "Profit", "Quantity", "Discount", "Shipping Duration"]
print(orders[num_cols].corr())

# 13. Returns Analysis
returns_unique = returns.drop_duplicates(subset=['Order ID'])
orders = orders.merge(returns_unique, on='Order ID', how='left')
orders['Return Status'] = orders['Returned'].apply(lambda x: 'Returned' if x == 'Yes' else 'Not Returned')
print("\n--- 13. Return Status Counts ---")
print(orders['Return Status'].value_counts())

# 14. Outlier Analysis (IQR)
q1 = orders['Sales'].quantile(0.25)
q3 = orders['Sales'].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
outliers = orders[(orders['Sales'] < lower_limit) | (orders['Sales'] > upper_limit)]
print(f"\n--- 14. Outlier Analysis ---")
print(f"IQR: {iqr:.2f} | Limits: [{lower_limit:.2f}, {upper_limit:.2f}] | Outliers Count: {len(outliers)}")

### 15. Data Visualizations (11 Charts)
Matplotlib and Seaborn were used to create 11 distinct charts for comprehensive exploration.

In [ ]:
# 1. Sales by Category
plt.figure(figsize=(6, 4))
orders.groupby('Category')['Sales'].sum().plot(kind='bar', color='#1f77b4')
plt.title('Sales by Category')
plt.ylabel('Sales')
plt.xticks(rotation=0)
plt.show()

# 2. Profit by Region
plt.figure(figsize=(6, 4))
orders.groupby('Region')['Profit'].sum().plot(kind='bar', color='#1f77b4')
plt.title('Profit by Region')
plt.ylabel('Profit')
plt.xticks(rotation=0)
plt.show()

# 3. Sales Trend by Year
plt.figure(figsize=(6, 4))
orders.groupby('Year')['Sales'].sum().plot(kind='line', marker='o', color='#1f77b4')
plt.title('Sales Trend by Year')
plt.ylabel('Sales')
plt.show()

# 4. Sales Performance
plt.figure(figsize=(6, 4))
orders['Sales Performance'].value_counts().sort_index().plot(kind='bar', color='#1f77b4')
plt.title('Sales Performance')
plt.xlabel('Sales Performance')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

# 5. Profit Margin by Category
plt.figure(figsize=(6, 4))
cat_margin = orders.groupby('Category').apply(lambda x: (x['Profit'].sum() / x['Sales'].sum()) * 100)
cat_margin.plot(kind='bar', color='#1f77b4')
plt.title('Profit Margin by Category')
plt.ylabel('Profit Margin (%)')
plt.xticks(rotation=0)
plt.show()

# 6. Discount vs Profit
plt.figure(figsize=(6, 4))
sns.boxplot(data=orders, x='Discount', y='Profit')
plt.title('Discount vs Profit')
plt.show()

# 7. Sales by Region
plt.figure(figsize=(6, 4))
orders.groupby('Region')['Sales'].sum().plot(kind='bar', color='#1f77b4')
plt.title('Sales by Region')
plt.ylabel('Sales')
plt.xticks(rotation=0)
plt.show()

# 8. Profit by Category
plt.figure(figsize=(6, 4))
orders.groupby('Category')['Profit'].sum().plot(kind='bar', color='#1f77b4')
plt.title('Profit by Category')
plt.ylabel('Profit')
plt.xticks(rotation=0)
plt.show()

# 9. Profit Trend by Year
plt.figure(figsize=(6, 4))
orders.groupby('Year')['Profit'].sum().plot(kind='line', marker='o', color='#1f77b4')
plt.title('Profit Trend by Year')
plt.ylabel('Profit')
plt.show()

# 10. Sales by Customer Segment
plt.figure(figsize=(6, 4))
orders.groupby('Segment')['Sales'].sum().plot(kind='bar', color='#1f77b4')
plt.title('Sales by Customer Segment')
plt.ylabel('Sales')
plt.xticks(rotation=0)
plt.show()

# 11. Returned vs Non-Returned Orders
plt.figure(figsize=(6, 4))
orders['Return Status'].value_counts().plot(kind='bar', color='#1f77b4')
plt.title('Returned vs Non-Returned Orders')
plt.ylabel('Number of Orders')
plt.xticks(rotation=0)
plt.show()

### 16. Dashboards
Simple analytical dashboards combining multiple charts:
* **Dashboard 1 — Executive Overview**: Sales by Category & Profit by Region.
* **Dashboard 2 — Sales Performance**: Sales Trend by Year & Sales Performance Categories.
* **Dashboard 3 — Profitability**: Profit Margin by Category & Discount vs Profit.
* **Dashboard 4 — Customers & Returns**: Sales by Customer Segment & Returned vs Non-Returned Orders.

In [ ]:
# Dashboard 1 — Executive Overview
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders.groupby('Category')['Sales'].sum().plot(kind='bar', ax=axes[0], color='#1f77b4', title='Sales by Category')
orders.groupby('Region')['Profit'].sum().plot(kind='bar', ax=axes[1], color='#1f77b4', title='Profit by Region')
plt.tight_layout()
plt.show()

# Dashboard 2 — Sales Performance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders.groupby('Year')['Sales'].sum().plot(kind='line', marker='o', ax=axes[0], color='#1f77b4', title='Sales Trend by Year')
orders['Sales Performance'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#1f77b4', title='Sales Performance')
plt.tight_layout()
plt.show()

# Dashboard 3 — Profitability
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cat_margin.plot(kind='bar', ax=axes[0], color='#1f77b4', title='Profit Margin by Category')
sns.boxplot(data=orders, x='Discount', y='Profit', ax=axes[1])
axes[1].set_title('Discount vs Profit')
plt.tight_layout()
plt.show()

# Dashboard 4 — Customers & Returns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders.groupby('Segment')['Sales'].sum().plot(kind='bar', ax=axes[0], color='#1f77b4', title='Sales by Customer Segment')
orders['Return Status'].value_counts().plot(kind='bar', ax=axes[1], color='#1f77b4', title='Returned vs Non-Returned Orders')
plt.tight_layout()
plt.show()

### 17. Main Business Insights
* **Technology** is one of the strongest categories in terms of sales and profit.
* **Furniture** has a significantly lower profit margin than Technology and Office Supplies.
* **West** is the strongest region overall; **Central** has the lowest regional profit margin.
* **Consumer** generates the highest sales, while **Home Office** has the highest profit margin.
* Higher discount levels are strongly associated with negative profitability.
* Sales generally increased from 2016 to 2019.
* The return rate is approximately 8%.

### 18. Conclusion & 19. Technologies Used
The complete pipeline covers:
`Data Import → Data Inspection → Data Cleaning → Feature Engineering → KPI Analysis → EDA → Statistical Analysis → Visualization → Dashboard → Business Insights`

| Technology | Purpose |
| :--- | :--- |
| **Python** | Main programming language |
| **Pandas** | Data manipulation and analysis |
| **NumPy** | Numerical operations |
| **Matplotlib** | Data visualization |
| **Seaborn** | Advanced statistical visualization |
| **Excel** | Source dataset |

---
**Name:** أدهم  
**Course:** Professional Data Analyst — Mini-Project 1